<p style="text-align: center">
<img src="../../../assets/images/dtlogo.png" alt="Duckietown" width="50%">
</p>

# Final Project

This is your **final project**. Unlike the earlier exercises, there is no skeleton to fill in line-by-line — you write the robot's behaviour from scratch.

Everything you build goes in one file:

```
tasks/project/packages/agent.py
```

Inside it you implement a single function:

```python
def main(camera, wheels, leds, stop_event):
    ...
```

When you deploy the task, the server hands you four ready-to-use objects and then runs your `main()` on its own thread. This notebook documents exactly what each of those four objects can do, then shows a minimal robot that drives forward.

## The contract

Your `main()` is called **once** with four arguments. You own the loop — keep running until you are told to stop, then leave the robot in a safe state.

| Argument | What it is | What you do with it |
|----------|-----------|---------------------|
| `camera` | the forward-facing camera | read frames |
| `wheels` | the two drive motors | set wheel speeds |
| `leds`   | the four RGB corner lights | set colors |
| `stop_event` | a `threading.Event` | check it to know when to exit |

The skeleton of every project looks like this:

```python
def main(camera, wheels, leds, stop_event):
    while not stop_event.is_set():     # loop until the server stops the task
        ok, frame = camera.read()      # get a camera frame
        if not ok:
            continue                   # no frame yet, try again
        # ... decide what to do based on `frame` ...
        wheels.set_wheels_speed(0.3, 0.3)

    wheels.set_wheels_speed(0.0, 0.0)  # IMPORTANT: stop the motors on exit
```

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

## 1. `camera` — reading frames

The Duckiebot has a forward-facing camera at **640×480**. The one call you need is:

```python
success, frame = camera.read()
```

- `success` is `True` when a frame was captured.
- `frame` is a NumPy array of shape `(height, width, 3)`, dtype `uint8`, in **BGR** order (this is OpenCV's default — Blue channel first, not Red).

Other handy attributes:

| Call | Returns |
|------|---------|
| `camera.read()` | `(success, frame)` — BGR image |
| `camera.resolution` | `(width, height)` |
| `camera.frame_count` | number of frames captured so far |

> **BGR vs RGB:** `camera.read()` gives you **BGR**. OpenCV functions expect BGR, so they work directly. But matplotlib and most ML models expect **RGB** — convert with `cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)` before using those.

In [ ]:
# A real frame looks like this. (We load a sample file here since the notebook
# isn't connected to the robot — on the bot you'd get this from camera.read().)
frame_rgb = plt.imread('../../../assets/samples/big-duck/big-duck-08.jpg')
print('shape:', frame_rgb.shape)   # (height, width, 3)
print('dtype:', frame_rgb.dtype)   # uint8

plt.imshow(frame_rgb)
plt.axis('off')
plt.title('A camera frame')
plt.show()

In [ ]:
# Reminder of the BGR/RGB difference. camera.read() returns BGR.
# If you hand a BGR frame straight to matplotlib, the colors come out wrong:
frame_bgr = cv2.cvtColor(frame_rgb, cv2.COLOR_RGB2BGR)  # simulate a camera.read() result

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
axes[0].imshow(frame_bgr)
axes[0].set_title('BGR shown as-is (wrong)')
axes[1].imshow(cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB))
axes[1].set_title('After BGR -> RGB (correct)')
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

## 2. `wheels` — driving

The robot uses **differential drive**: you set each wheel's speed independently and it steers by spinning them at different rates.

```python
wheels.set_wheels_speed(left, right)
```

Each speed is in `[-1.0, 1.0]` — positive is forward, negative is backward. The robot turns **toward the slower wheel**.

| Left | Right | Result |
|------|-------|--------|
| +0.5 | +0.5 | Straight forward |
| -0.5 | -0.5 | Straight backward |
| +0.5 | -0.5 | Spin right in place |
| -0.5 | +0.5 | Spin left in place |
| +0.5 | +0.2 | Curve right |

> **Always stop the motors before `main()` returns** with `wheels.set_wheels_speed(0.0, 0.0)`. If you don't, the robot keeps driving after the task ends.

In [ ]:
examples = [
    (0.5,  0.5,  'Forward'),
    (-0.5, -0.5, 'Backward'),
    (0.5,  -0.5, 'Spin right'),
    (0.5,  0.2,  'Curve right'),
]

fig, axes = plt.subplots(1, 4, figsize=(10, 3))
for ax, (left, right, label) in zip(axes, examples):
    ax.bar(['Left', 'Right'], [left, right], color=['#1f6feb', '#e67e22'])
    ax.set_ylim(-1.1, 1.1)
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_title(label)
    ax.set_ylabel('Speed')
plt.tight_layout()
plt.show()

## 3. `leds` — corner lights

Four RGB LEDs sit at the corners. **Note the indices skip 1** — they are `0, 2, 3, 4`:

```
          FRONT
    [0] ───────── [2]
     |             |
    [3] ───────── [4]
          BACK
```

```python
leds.set_rgb(index, [r, g, b])   # each component 0.0 .. 1.0
```

| Call | Effect |
|------|--------|
| `leds.set_rgb(0, [1, 0, 0])` | front-left red |
| `leds.set_rgb(2, [0, 1, 0])` | front-right green |
| `leds.all_on()` | all four white |
| `leds.all_off()` | all four off |

Common colors: `[1,0,0]` red, `[0,1,0]` green, `[0,0,1]` blue, `[1,1,0]` yellow, `[1,1,1]` white, `[0,0,0]` off.

> `leds` may be `None` if the LED hardware failed to initialize. Guard with `if leds:` before using it so your project still runs without lights.

In [ ]:
# Visualize the 4 corner LEDs set to a 'hazard' pattern (all yellow)
colors = {0: [1, 1, 0], 2: [1, 1, 0], 3: [1, 1, 0], 4: [1, 1, 0]}
positions = {0: (0, 1), 2: (1, 1), 3: (0, 0), 4: (1, 0)}
labels = {0: 'LED 0\nfront-left', 2: 'LED 2\nfront-right',
          3: 'LED 3\nback-left',  4: 'LED 4\nback-right'}

fig, ax = plt.subplots(figsize=(3.5, 3.5))
for idx, (col, row) in positions.items():
    ax.add_patch(plt.Rectangle((col, row), 0.9, 0.9, color=colors[idx]))
    ax.text(col + 0.45, row + 0.45, labels[idx], ha='center', va='center', fontsize=8)
ax.set_xlim(-0.1, 2); ax.set_ylim(-0.1, 2)
ax.set_aspect('equal'); ax.axis('off')
ax.set_title('leds.set_rgb(i, [1, 1, 0])')
plt.show()

## 4. `stop_event` — knowing when to quit

`stop_event` is a standard `threading.Event`. The server sets it when the task is being shut down (you press Stop, or the process is killed). Your loop must check it:

```python
while not stop_event.is_set():
    # do work
```

You can also wait instead of busy-looping — `stop_event.wait(0.1)` sleeps for up to 0.1s and returns early if the event fires. If you ignore `stop_event`, your thread never exits and the robot won't stop cleanly.

## 5. Putting it together

Here is a complete, minimal `main()` — a robot that drives forward, flashes its front LEDs green, and stops cleanly. This is the shape your project should take. Copy it into `tasks/project/packages/agent.py` as a starting point.

```python
import time
import cv2

def main(camera, wheels, leds, stop_event):
    # set up: front LEDs green
    if leds:
        leds.set_rgb(0, [0.0, 1.0, 0.0])
        leds.set_rgb(2, [0.0, 1.0, 0.0])

    try:
        while not stop_event.is_set():
            ok, frame = camera.read()        # BGR frame
            if not ok:
                time.sleep(0.02)
                continue

            # rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)  # if a model needs RGB
            # ... your perception + decision logic here ...

            wheels.set_wheels_speed(0.3, 0.3)  # drive forward
    finally:
        # always runs, even on error or stop
        wheels.set_wheels_speed(0.0, 0.0)
        if leds:
            leds.all_off()
```

The `try/finally` guarantees the motors stop and the lights turn off no matter how the loop ends.

## 6. Running your project

The project runs on **real hardware** (there is no simulation scene for it). From your laptop:

```bash
# deploy your agent.py and start it on the bot
python launch.py --run --bot <bot-name> --task project

# stop it when you're done
python launch.py --stop --bot <bot-name>
```

While it runs, open the web UI at `http://<bot-name>.local:5000` to watch the camera feed live.

Each time you change `agent.py`, re-run the `--run` command — it re-packages your file and restarts the task on the bot.

## 7. Your assignment

Combine what you learned across the earlier modules into one autonomous behaviour. Some directions:

- **Lane following** — use the image-filtering and visual-servoing techniques to keep the bot in its lane.
- **Object reaction** — load your trained detector and stop / steer when you see a duck, truck, or sign.
- **Signal with LEDs** — flash the corner lights to indicate turns or a detected obstacle.
- **Braitenberg-style** — react directly to brightness in the frame, no training needed.

Whatever you build, the only requirement is the contract: implement `main(camera, wheels, leds, stop_event)`, loop on `stop_event`, and stop the motors when you exit.

Good luck! 🦆